# 01 — Exploración de Datos de Siniestros
**HackIAthon 2026 | Reto Aseguradora del Sur — FraudIA Claims**

Este notebook explora el dataset sintético de 500 siniestros para entender la distribución de variables, señales de riesgo y patrones relevantes para la detección de posible fraude.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

DATA_PATH = Path('../backend/ai_data_core/data/synthetic/siniestros_scored.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

## 1. Vista general del dataset

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print('\n=== Valores nulos ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Estadísticas descriptivas (numéricas) ===')
df.describe()

## 2. Distribución del Score de Riesgo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma del score
axes[0].hist(df['score_riesgo'], bins=20, color='#3b82f6', edgecolor='white')
axes[0].set_title('Distribución del Score de Riesgo')
axes[0].set_xlabel('Score (0-100)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(40, color='green',  linestyle='--', label='Límite VERDE/AMARILLO (40)')
axes[0].axvline(75, color='orange', linestyle='--', label='Límite AMARILLO/ROJO (75)')
axes[0].legend()

# Semáforo de riesgo
semaforo = df['nivel_riesgo'].value_counts()
colores  = {'VERDE': '#22c55e', 'AMARILLO': '#f59e0b', 'ROJO': '#ef4444'}
axes[1].pie(
    semaforo.values,
    labels=semaforo.index,
    colors=[colores.get(n, 'gray') for n in semaforo.index],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Distribución Semáforo de Riesgo')
plt.tight_layout()
plt.savefig('distribucion_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nConteo por nivel de riesgo:')
print(semaforo)

## 3. Análisis por Ramo y Cobertura

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ramo_riesgo = df.groupby('ramo')['score_riesgo'].mean().sort_values(ascending=False)
ramo_riesgo.plot(kind='bar', ax=axes[0], color='#6366f1')
axes[0].set_title('Score Promedio por Ramo')
axes[0].set_xlabel('Ramo')
axes[0].set_ylabel('Score promedio')
axes[0].tick_params(axis='x', rotation=30)

cobertura_count = df[df['nivel_riesgo'] == 'ROJO']['cobertura'].value_counts().head(8)
cobertura_count.plot(kind='barh', ax=axes[1], color='#ef4444')
axes[1].set_title('Top 8 Coberturas en Casos ROJOS')
axes[1].set_xlabel('Cantidad de casos')
plt.tight_layout()
plt.show()

## 4. Señales de Fraude — Análisis Univariado

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Días desde inicio de póliza
for nivel, color in [('VERDE','#22c55e'), ('AMARILLO','#f59e0b'), ('ROJO','#ef4444')]:
    axes[0,0].hist(
        df[df['nivel_riesgo']==nivel]['dias_desde_inicio_poliza'].clip(0, 365),
        bins=20, alpha=0.6, label=nivel, color=color
    )
axes[0,0].set_title('Días desde inicio de póliza')
axes[0,0].legend()

# Días entre ocurrencia y reporte
for nivel, color in [('VERDE','#22c55e'), ('AMARILLO','#f59e0b'), ('ROJO','#ef4444')]:
    axes[0,1].hist(
        df[df['nivel_riesgo']==nivel]['dias_entre_ocurrencia_reporte'].clip(0, 30),
        bins=15, alpha=0.6, label=nivel, color=color
    )
axes[0,1].set_title('Días entre ocurrencia y reporte')
axes[0,1].legend()

# Historial de siniestros
hist_risk = df.groupby(['historial_siniestros_asegurado', 'nivel_riesgo']).size().unstack(fill_value=0)
hist_risk.plot(kind='bar', ax=axes[0,2], color=['#22c55e','#ef4444','#f59e0b'])
axes[0,2].set_title('Historial vs Nivel de riesgo')
axes[0,2].set_xlabel('Siniestros previos')

# Monto reclamado
df.boxplot(column='monto_reclamado', by='nivel_riesgo', ax=axes[1,0])
axes[1,0].set_title('Monto reclamado por nivel')
axes[1,0].set_xlabel('Nivel de riesgo')

# Documentos incompletos
doc_risk = df.groupby(['documentos_completos', 'nivel_riesgo']).size().unstack(fill_value=0)
doc_risk.index = ['Incompletos', 'Completos']
doc_risk.plot(kind='bar', ax=axes[1,1], color=['#22c55e','#ef4444','#f59e0b'])
axes[1,1].set_title('Documentos vs Nivel de riesgo')
axes[1,1].tick_params(axis='x', rotation=0)

# Doc inconsistente
inc_risk = df.groupby(['tiene_doc_inconsistente', 'nivel_riesgo']).size().unstack(fill_value=0)
inc_risk.index = ['Sin inconsistencia', 'Con inconsistencia']
inc_risk.plot(kind='bar', ax=axes[1,2], color=['#22c55e','#ef4444','#f59e0b'])
axes[1,2].set_title('Inconsistencia documental vs Riesgo')
axes[1,2].tick_params(axis='x', rotation=15)

plt.suptitle('Señales de posible fraude por nivel de riesgo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('senales_fraude.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análisis de Montos

In [ ]:
print('=== Montos por nivel de riesgo ($USD) ===')
resumen_montos = df.groupby('nivel_riesgo')['monto_reclamado'].agg(['mean','median','max','sum'])
resumen_montos.columns = ['Promedio', 'Mediana', 'Máximo', 'Total']
resumen_montos = resumen_montos.applymap(lambda x: f'${x:,.0f}')
print(resumen_montos)

print('\n=== Correlación entre score y variables numéricas ===')
num_cols = ['score_riesgo', 'monto_reclamado', 'monto_estimado',
            'dias_desde_inicio_poliza', 'dias_entre_ocurrencia_reporte',
            'historial_siniestros_asegurado']
corr = df[num_cols].corr()['score_riesgo'].sort_values(ascending=False)
print(corr)

## 6. Top 10 Siniestros de Mayor Riesgo

In [ ]:
top10 = df.nlargest(10, 'score_riesgo')[[
    'id_siniestro', 'ramo', 'cobertura', 'monto_reclamado',
    'score_riesgo', 'nivel_riesgo', 'historial_siniestros_asegurado'
]]
top10['monto_reclamado'] = top10['monto_reclamado'].apply(lambda x: f'${x:,.0f}')
print('TOP 10 Siniestros con Mayor Score de Riesgo:')
top10

## 7. Conclusiones

- Los siniestros **ROJOS** concentran montos más altos y mayor historial de reclamos del asegurado.
- La variable más correlacionada con el score es **`historial_siniestros_asegurado`**.
- Los casos con **documentos inconsistentes** tienen una tasa de riesgo ROJO significativamente más alta.
- El ramo de **Vehículos** concentra el mayor número de casos sospechosos por volumen.
- Los siniestros reportados con **>7 días de demora** tienen score promedio superior al resto.

> ⚠️ Este análisis es referencial. La decisión final de revisión siempre corresponde al analista humano.